# Residual Stream Statistics

Detailed statistical analysis of the residual stream: norm profile,
direction drift, variance decomposition, position similarity, and component balance.

## Why This Matters

Residual stream stream statistics characterizes the shared information bus that all transformer components read from and write to. Because the residual stream carries all inter-component communication, understanding its structure is fundamental to understanding the model as a whole.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [nostalgebraist (2020) "interpreting GPT: the logit lens"](https://www.lesswrong.com/posts/AcKRB8wDpdaN6v6ru/interpreting-gpt-the-logit-lens) — Project intermediate residuals through the unembedding

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.residual_stream_statistics import (
    residual_norm_profile, residual_direction_drift,
    residual_variance_decomposition, residual_position_similarity,
    residual_component_balance,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Norm Profile

Track how the residual norm grows through the network.

In [ ]:
result = residual_norm_profile(model, tokens)
print(f"Embed norm: {result['embed_norm']:.4f}")
print(f"Final norm: {result['final_norm']:.4f} (growth: {result['total_growth']:+.4f})\n")
for p in result['per_layer']:
    bar = '█' * int(p['mean_norm'] * 2)
    print(f"  Layer {p['layer']}: norm={p['mean_norm']:.4f}, "
          f"growth={p['norm_growth']:+.4f} ({p['growth_rate']:+.1%}) {bar}")

## Direction Drift

How much does the residual direction change between layers?

In [ ]:
result = residual_direction_drift(model, tokens)
print(f"Mean drift: {result['mean_drift']:.4f}\n")
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: cos_to_prev={p['cos_to_previous']:+.4f}, "
          f"cos_to_embed={p['cos_to_embed']:+.4f}")

## Variance Decomposition

How much variance comes from attention vs MLP?

In [ ]:
result = residual_variance_decomposition(model, tokens)
for p in result['per_layer']:
    attn_bar = '█' * int(p['attn_fraction'] * 20)
    mlp_bar = '░' * int(p['mlp_fraction'] * 20)
    print(f"  Layer {p['layer']}: resid_var={p['residual_variance']:.6f}, "
          f"attn={p['attn_fraction']:.1%} mlp={p['mlp_fraction']:.1%} {attn_bar}{mlp_bar}")

## Position Similarity

Are residual vectors similar across positions, or diverse?

In [ ]:
for layer in range(4):
    result = residual_position_similarity(model, tokens, layer=layer)
    conv = 'CONVERGED' if result['is_converged'] else 'diverse'
    print(f"Layer {layer}: mean_sim={result['mean_similarity']:.4f} [{conv}]")
    for p in result['pairs'][:3]:
        print(f"    pos{p['pos_a']}↔pos{p['pos_b']}: {p['similarity']:+.4f}")
    print()

## Component Balance

Attention vs MLP: relative magnitude, alignment, dominance.

In [ ]:
result = residual_component_balance(model, tokens)
print(f"Attn dominant: {result['n_attn_dominant']}, MLP dominant: {result['n_mlp_dominant']}\n")
for p in result['per_layer']:
    coop = '+' if p['is_cooperative'] else '-'
    print(f"  Layer {p['layer']}: attn={p['attn_norm']:.4f}, "
          f"mlp={p['mlp_norm']:.4f}, ratio={p['attn_mlp_ratio']:.2f}, "
          f"align={p['alignment']:+.4f} [{p['dominant']}] [{coop}]")